# OSM Notes Classifier: training and results

OpenStreetMap notes are messages that
people leave on the map: "this shop closed", "this road does not exist",
"wrong street name". There are more than **5.4 million** notes, and mappers
have to read them one by one to know what to do.

This notebook trains a model that reads a note and gives it two labels:

- **Category**: what the mapper should do.
- **Theme**: what the note is about (food, roads, health...).

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Rub21/osm-notes-deploy/blob/main/notebooks/train_colab.ipynb)
Code and data pipeline: [Rub21/osm-notes-deploy](https://github.com/Rub21/osm-notes-deploy).

### Categories

| Category | Meaning | Example |
|---|---|---|
| **Needs a map change** | | |
| `missing_place` | A place or POI exists and is not on the map | "Missing bakery on this corner" |
| `place_changed` | A mapped place closed, moved or is now something else | "This shop is now a restaurant" |
| `wrong_name` | The name of a place or street is wrong or missing | "It is Grau Ave, not Brasil Ave" |
| `wrong_road` | Roads, paths or tracks that are missing, wrong, or have wrong attributes | "This street is a dead end" |
| `other_data` | Other data: opening hours, address, phone, type, land use | "The opening hours changed" |
| **Close without a change** | | |
| `spam` | Ads or unrelated links | "Best chicken in town, call 999..." |
| `empty` | Says nothing: empty, greetings, tests | "ok", "asdf" |
| `personal_data` | Reveals where a private person lives, or their data. The note should be hidden | "This is my house" |
| `not_mappable` | Opinions, complaints, or info that does not go in OSM | "Dangerous area" |
| **Doubt** | | |
| `unsure` | Not clear. Not used for training: doubt comes from the model's confidence | |

**Themes:** `food`, `shop`, `health`, `education`, `transport`, `roads`,
`buildings`, `leisure`, `nature`, `services`, `none`.

### Data

- All notes come from the OSM planet notes dump, updated every minute.
- Only the **first comment** of each note is used (the closing comments would
  give away the answer).
- About 31% of notes come from apps (MAPS.ME, Organic Maps, CoMaps,
  StreetComplete, OnOSM) with a fixed template. The template text is removed
  and the model reads a short tag plus the person's own words, for example
  `[app_place_gone] [amenity-pharmacy] Аптеку закрыли`.
- Labels: Spanish and English notes, **pre-labeled by AI** and partly
  **reviewed by people** on the
  [labeling page](https://osm-notes-labels.rub21.com). All labels are mixed
  and used the same way. The model only reads the note text, never who
  labeled it.

### Models

| Model | Size | Notes |
|---|---|---|
| TF-IDF + logistic regression | – | Baseline. If a transformer does not beat it, the problem is in the data |
| [`microsoft/mdeberta-v3-base`](https://huggingface.co/microsoft/mdeberta-v3-base) | 280M | Fast, multilingual |
| [`FacebookAI/xlm-roberta-large`](https://huggingface.co/FacebookAI/xlm-roberta-large) | 560M | Strongest |

One model for category and one for theme. Everything is saved in Google Drive
with checkpoints: if Colab disconnects, run all cells again and it continues
from the last checkpoint.

### How to run

1. **Runtime → Change runtime type → GPU** (A100 or L4; T4 works but is slower).
2. **Runtime → Run all.** It asks for access to Google Drive.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Install libraries

In [ ]:
!pip install -q "transformers>=4.46,<5" "accelerate>=1.1" "datasets>=3.1" sentencepiece protobuf

## 2. Settings

- `FILTERS`: API filters, for example `lang=es,en&since=2026-10-30`. Also
  `user=<name>`, `category=spam,personal_data`, `theme=food`, `changed=1`
  (only labels where a person changed the AI suggestion).
- `MODELS`: models to train. Each one is trained and measured on its own.
  `intfloat/multilingual-e5-large` (560M) is an optional third one.
- `OUT`: where training writes, on the Colab disk (fast, does not fill Drive).
  At the end, models and results (no checkpoints) are copied to `DRIVE_OUT`.
- `RUN_NAME`: same name = resume; new name = start from zero. The train/test
  split is saved the first time and reused when resuming.
- `SAVE_STEPS`: evaluate and save a checkpoint every N steps. The best model
  ends in `model/`.
- `EPOCHS` is a maximum: early stopping (`PATIENCE`) ends training when
  macro-F1 stops improving.
- `KEEP_CHECKPOINTS`: `False` deletes the checkpoints when a model is done
  (~3 GB each for base models, ~6.5 GB for large).

In [ ]:
import os
os.environ["WANDB_MODE"] = "disabled"  # no W&B prompt

from google.colab import drive
drive.mount("/content/drive")

FILTERS = ""
LABELS_URL = "https://osm-notes-labels.rub21.com/api/labels.csv"

MODELS = [
    "microsoft/mdeberta-v3-base",
    "FacebookAI/xlm-roberta-large",
    # "intfloat/multilingual-e5-large",
]
TRAIN_LANGS = {"es", "en", "und"}
TEST_SIZE = 0.2      # 20% of the notes for testing
VAL_SIZE = 0.1       # 10% of the rest to pick the best checkpoint
MIN_PER_CLASS = 8    # classes with fewer labels are left out
MAX_LEN = 128        # tokens per note
EPOCHS = 10          # max; early stopping ends before
BATCH_SIZE = 32
LR_BASE = 3e-5       # learning rate for base models
LR_LARGE = 2e-5      # learning rate for large models
PATIENCE = 3         # stop if macro-F1 does not improve in 3 evaluations
SAVE_STEPS = 400     # evaluate and save a checkpoint every 400 steps
KEEP_CHECKPOINTS = False
SEED = 42
RUN_NAME = "ft_v2_context"
# RUN_NAME = "ft_20261109"
OUT = "/content/out"   # Colab disk while training
DRIVE_OUT = "/content/drive/MyDrive/osm-notes-training"  # final copy

## 3. Download the labels

In [ ]:
import urllib.request
import pandas as pd

url = LABELS_URL + (f"?{FILTERS}" if FILTERS else "")
# Cloudflare blocks the default "Python-urllib" user agent (403).
req = urllib.request.Request(url, headers={"User-Agent": "osm-notes-trainer/1.0"})
with urllib.request.urlopen(req) as r, open("/content/labels.csv", "wb") as f:
    f.write(r.read())

labels = pd.read_csv("/content/labels.csv", encoding="utf-8-sig", dtype=str, keep_default_na=False)
labels = labels.replace({"": None})
labels["note_id"] = labels["note_id"].astype(int)
print(len(labels), "labels from", url)
labels.head()

## 4. Data overview

How many labels there are per class, language and template. A class with
fewer than 20 labels is hard to learn.

In [ ]:
import matplotlib.pyplot as plt
plt.rcParams["figure.dpi"] = 110

df = labels.copy()
df["template"] = df["template"].fillna("(no template)")
df["user_text"] = df["user_text"].fillna("")
df["words"] = df["user_text"].str.split().str.len()
print(f"Labels: {len(df)} · notes: {df.note_id.nunique()} · labelers: {df.user_name.nunique()}")

fig, ax = plt.subplots(2, 3, figsize=(20, 10))
def bars(s, a, title, min_line=None):
    s.sort_values().plot.barh(ax=a, title=title)
    for i, v in enumerate(s.sort_values()):
        a.text(v, i, f" {v}", va="center", fontsize=8)
    if min_line:
        a.axvline(min_line, color="red", ls="--", lw=1)
bars(df["category"].value_counts(), ax[0, 0], "By category (red = 20)", 20)
bars(df["theme"].fillna("(empty)").value_counts(), ax[0, 1], "By theme (red = 20)", 20)
bars(df["lang"].value_counts().head(15), ax[0, 2], "By language")
bars(df["template"].value_counts(), ax[1, 0], "By app template")
bars(df["user_name"].value_counts().head(15), ax[1, 1], "By labeler")
df["words"].clip(upper=60).plot.hist(ax=ax[1, 2], bins=30, title="Words written by the person (60+ grouped)")
plt.tight_layout(); plt.show()

In [ ]:
# Category x theme: which themes appear in each category.
ct = pd.crosstab(df["category"], df["theme"].fillna("(empty)"))
fig, a = plt.subplots(figsize=(14, 6))
a.imshow(ct.values, cmap="Greens")
a.set_xticks(range(len(ct.columns)), ct.columns, rotation=45, ha="right")
a.set_yticks(range(len(ct.index)), ct.index)
for i in range(ct.shape[0]):
    for j in range(ct.shape[1]):
        if ct.values[i, j]:
            a.text(j, i, ct.values[i, j], ha="center", va="center", fontsize=8)
a.set_title("Category x theme"); plt.tight_layout(); plt.show()

# Length of the person's text (words), per category.
display(df.groupby("category")["words"].describe()[["count", "mean", "50%", "max"]].round(1))

### How often does the final label keep the first AI suggestion?

`claude_category` / `claude_theme` is the first AI suggestion. The final label
can differ because a person reviewed it or because several AI models voted.
The table below shows what changed (row = first suggestion, column = final label).

In [ ]:
hum = df[df["claude_category"].notna()]
if len(hum):
    ok_cat = (hum["category"] == hum["claude_category"]).mean()
    ok_th = (hum["theme"] == hum["claude_theme"]).mean()
    print(f"Labels with an AI suggestion: {len(hum)} · category match {ok_cat:.1%} · theme match {ok_th:.1%}")
    per = hum.assign(ok=hum["category"] == hum["claude_category"]).groupby("claude_category")["ok"] \
             .agg(["count", "mean"]).rename(columns={"count": "n", "mean": "match"}).sort_values("match")
    display(per.style.format({"match": "{:.0%}"}))
    changed = hum[hum["category"] != hum["claude_category"]]
    if len(changed):
        print("What changed (row = first AI suggestion, column = final label):")
        display(pd.crosstab(changed["claude_category"], changed["category"]))
else:
    print("No labels with an AI suggestion.")

## 5. Training code

1. One label per note: if several labelers labeled it, the majority wins;
   on a tie, the newest label wins (a correction beats the first label).
   `unsure` is not used for category.
2. Model input: `[template] [poi_type] [context] user_text`. `context` is the
   short app context (StreetComplete question and element, POI name). Only the
   text, not who labeled it.
3. 72% train, 8% validation (picks the best checkpoint), 20% test, stratified
   by class. Languages outside `TRAIN_LANGS` go to `measure_only`.
4. Loss with class weights (square root of "balanced"), so rare classes like
   `personal_data` and `spam` count more.
5. Saves to `OUT/RUN_NAME/<target>/`:
   - `split.csv`: which note went to each set;
   - `<model>/model/`: the best model, ready for `from_pretrained`;
   - `<model>/checkpoints/`: to resume;
   - `<model>/predictions.csv`, `metrics.json`, `log_history.json`;
   - `report.md`: all models compared.

In [ ]:
import datetime as dt
import gc
import json
import shutil
from collections import Counter

import numpy as np
import torch
from datasets import Dataset
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, f1_score
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from transformers import (AutoModelForSequenceClassification, AutoTokenizer,
                          DataCollatorWithPadding, EarlyStoppingCallback, Trainer,
                          TrainingArguments)

EVAL_SPLITS = ["test", "measure_only"]


def model_input(r) -> str:
    """Short template tags + app context + the user's words."""
    tags = "".join(f"[{v}] " for v in (r.template, r.poi_type, r.context) if isinstance(v, str) and v)
    return (tags + " ".join(str(r.user_text or "").split())).strip()


def prepare(labels: pd.DataFrame, target: str) -> pd.DataFrame:
    """One row per note with the majority label."""
    df = labels[labels[target].notna()]
    if target == "category":
        df = df[df[target] != "unsure"]  # "I don't know" teaches nothing

    def vote(g: pd.DataFrame) -> pd.Series:
        g = g.sort_values("updated_at", ascending=False)  # on a tie, the newest label wins
        label, n = Counter(g[target]).most_common(1)[0]
        first = g.iloc[0]
        return pd.Series({
            "label": label, "votes": len(g), "agreement": n / len(g),
            "claude": first[f"claude_{target}"], "lang": first["lang"],
            "template": first["template"], "poi_type": first["poi_type"],
            "context": first.get("context"),
            "user_text": first["user_text"], "user_name": ",".join(sorted(set(g["user_name"])))})

    out = df.groupby("note_id").apply(vote, include_groups=False).reset_index()
    out["text"] = [model_input(r) for r in out.itertuples(index=False)]
    return out


def split(df: pd.DataFrame, split_file: str) -> pd.DataFrame:
    """Assign train/val/test/measure_only. Saved once, reused on resume."""
    if os.path.isfile(split_file):
        saved = pd.read_csv(split_file)
        data = df.merge(saved, on="note_id", how="inner")
        print(f"Saved split: {split_file} ({len(df) - len(data)} new notes left out)")
        return data

    pool = df[df["lang"].isin(TRAIN_LANGS)]
    counts = pool["label"].value_counts()
    small = sorted(counts[counts < MIN_PER_CLASS].index)
    if small:
        print(f"Left out (fewer than {MIN_PER_CLASS} labels): {', '.join(small)}")
    pool = pool[~pool["label"].isin(small)]
    classes = set(pool["label"])
    rest, test = train_test_split(pool, test_size=TEST_SIZE, random_state=SEED, stratify=pool["label"])
    train, val = train_test_split(rest, test_size=VAL_SIZE, random_state=SEED, stratify=rest["label"])
    measure_only = df[~df["lang"].isin(TRAIN_LANGS) & df["label"].isin(classes)]
    data = pd.concat([train.assign(split="train"), val.assign(split="val"), test.assign(split="test"),
                      measure_only.assign(split="measure_only")], ignore_index=True)
    data[["note_id", "split"]].to_csv(split_file, index=False)
    return data


def baseline(train: pd.DataFrame, ev: pd.DataFrame):
    """TF-IDF on character n-grams + logistic regression.
    If a transformer does not beat this, the problem is in the data."""
    vec = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), min_df=2, sublinear_tf=True)
    clf = LogisticRegression(max_iter=2000, class_weight="balanced", random_state=SEED)
    clf.fit(vec.fit_transform(train["text"]), train["label"])
    proba = clf.predict_proba(vec.transform(ev["text"]))
    return clf.classes_[proba.argmax(1)], proba.max(1)


def last_checkpoint(folder: str):
    if not os.path.isdir(folder):
        return None
    steps = [d for d in os.listdir(folder)
             if d.startswith("checkpoint-") and os.path.isfile(os.path.join(folder, d, "trainer_state.json"))]
    return os.path.join(folder, max(steps, key=lambda d: int(d.split("-")[1]))) if steps else None


def amp_flags(model_name: str) -> dict:
    if not torch.cuda.is_available():
        return {}
    if torch.cuda.is_bf16_supported():  # A100, L4
        return {"bf16": True}
    return {} if "deberta" in model_name else {"fp16": True}  # DeBERTa-v3 gives NaN with fp16


class WeightedTrainer(Trainer):
    """Cross-entropy with class weights."""

    def __init__(self, *args, class_weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits.float()
        cw = self.class_weights.to(logits.device) if self.class_weights is not None else None
        loss = torch.nn.functional.cross_entropy(logits, labels, weight=cw)
        return (loss, outputs) if return_outputs else loss


def eval_metrics(p) -> dict:
    logits = p.predictions[0] if isinstance(p.predictions, tuple) else p.predictions
    pred = logits.argmax(-1)
    return {"macro_f1": f1_score(p.label_ids, pred, average="macro", zero_division=0),
            "accuracy": accuracy_score(p.label_ids, pred)}


def finetune(model_name: str, train, val, ev, classes, out_dir: str):
    """Fine-tune (or load if done) and predict ev. Returns (pred, confidence)."""
    final_dir = os.path.join(out_dir, "model")
    ckpt_dir = os.path.join(out_dir, "checkpoints")
    done = os.path.isfile(os.path.join(final_dir, "config.json"))
    label2id = {c: i for i, c in enumerate(classes)}
    id2label = {i: c for c, i in label2id.items()}

    tok = AutoTokenizer.from_pretrained(final_dir if done else model_name)

    def to_ds(d: pd.DataFrame) -> Dataset:
        enc = tok(d["text"].tolist(), truncation=True, max_length=MAX_LEN)
        return Dataset.from_dict({"input_ids": enc["input_ids"], "attention_mask": enc["attention_mask"],
                                  "labels": [label2id[l] for l in d["label"]]})

    y = np.array([label2id[l] for l in train["label"]])
    cw = np.sqrt(compute_class_weight("balanced", classes=np.arange(len(classes)), y=y))
    cw = torch.tensor(cw / cw.mean(), dtype=torch.float)

    lr = LR_LARGE if "large" in model_name else LR_BASE
    args = TrainingArguments(
        output_dir=ckpt_dir, num_train_epochs=EPOCHS, learning_rate=lr,
        per_device_train_batch_size=BATCH_SIZE, per_device_eval_batch_size=BATCH_SIZE * 2,
        warmup_ratio=0.1, weight_decay=0.01, lr_scheduler_type="linear",
        eval_strategy="steps", eval_steps=SAVE_STEPS, save_strategy="steps", save_steps=SAVE_STEPS,
        save_total_limit=2, load_best_model_at_end=True, metric_for_best_model="macro_f1",
        greater_is_better=True, logging_steps=50, report_to="none", seed=SEED,
        dataloader_num_workers=2, **amp_flags(model_name))

    if done:
        print(f"Model already trained, loading: {final_dir}")
        model = AutoModelForSequenceClassification.from_pretrained(final_dir)
    else:
        model = AutoModelForSequenceClassification.from_pretrained(
            model_name, num_labels=len(classes), id2label=id2label, label2id=label2id)

    trainer = WeightedTrainer(
        model=model, args=args, train_dataset=to_ds(train), eval_dataset=to_ds(val),
        data_collator=DataCollatorWithPadding(tok), compute_metrics=eval_metrics,
        class_weights=cw, callbacks=[EarlyStoppingCallback(early_stopping_patience=PATIENCE)])

    if not done:
        ckpt = last_checkpoint(ckpt_dir)
        print(f"Resuming from {ckpt}" if ckpt else f"Training {model_name} (lr={lr}) ...", flush=True)
        trainer.train(resume_from_checkpoint=ckpt)
        trainer.save_model(final_dir)  # best checkpoint (load_best_model_at_end)
        tok.save_pretrained(final_dir)
        with open(os.path.join(out_dir, "log_history.json"), "w") as f:
            json.dump(trainer.state.log_history, f, indent=1)
        print(f"Best checkpoint: {trainer.state.best_model_checkpoint} "
              f"(macro-F1 val {trainer.state.best_metric:.3f}) -> {final_dir}")
        if not KEEP_CHECKPOINTS:
            shutil.rmtree(ckpt_dir, ignore_errors=True)

    logits = trainer.predict(to_ds(ev)).predictions
    logits = logits[0] if isinstance(logits, tuple) else logits
    proba = torch.softmax(torch.tensor(logits, dtype=torch.float), -1).numpy()
    pred = np.array([id2label[i] for i in proba.argmax(1)])

    del trainer, model
    gc.collect()
    torch.cuda.empty_cache()
    return pred, proba.max(1)


def scores(y_true, y_pred) -> dict:
    return {"n": int(len(y_true)),
            "accuracy": round(float(accuracy_score(y_true, y_pred)), 3) if len(y_true) else None,
            "macro_f1": round(float(f1_score(y_true, y_pred, average="macro", zero_division=0)), 3)
            if len(y_true) else None}


def per_class(t: pd.DataFrame) -> pd.DataFrame:
    rep = classification_report(t["label"], t["pred"], output_dict=True, zero_division=0)
    return pd.DataFrame([
        {"class": k, "precision": round(v["precision"], 2), "recall": round(v["recall"], 2),
         "f1": round(v["f1-score"], 2), "support": int(v["support"])}
        for k, v in rep.items() if k not in ("accuracy", "macro avg", "weighted avg")])


def md_table(df: pd.DataFrame) -> str:
    lines = ["| " + " | ".join(map(str, df.columns)) + " |", "|" + "---|" * len(df.columns)]
    lines += ["| " + " | ".join(str(v) for v in row) + " |" for row in df.itertuples(index=False)]
    return "\n".join(lines)


def short(model_name: str) -> str:
    return model_name.split("/")[-1]


def train_target(target: str) -> str:
    """Baseline + every model in MODELS for one target. Returns the target folder."""
    tdir = os.path.join(OUT, RUN_NAME, target)
    os.makedirs(tdir, exist_ok=True)
    df = prepare(labels, target)
    data = split(df, os.path.join(tdir, "split.csv"))
    train, val = data[data["split"] == "train"], data[data["split"] == "val"]
    ev = data[data["split"].isin(EVAL_SPLITS)].reset_index(drop=True)
    classes = sorted(train["label"].unique())
    print(f"== {target}: {len(classes)} classes · " +
          " · ".join(f"{s}={n}" for s, n in data["split"].value_counts().items()))

    runs = {"baseline": None, **{short(m): m for m in MODELS}}
    summary = []
    for name, model_name in runs.items():
        mdir = os.path.join(tdir, name)
        os.makedirs(mdir, exist_ok=True)
        pred_file = os.path.join(mdir, "predictions.csv")
        if os.path.isfile(pred_file):
            print(f"-- {name}: already done, skipping")
            out = pd.read_csv(pred_file)
        else:
            print(f"-- {name}", flush=True)
            pred, conf = baseline(train, ev) if model_name is None else \
                finetune(model_name, train, val, ev, classes, mdir)
            out = ev.assign(pred=pred, conf=np.round(conf, 4))
            out.to_csv(pred_file, index=False, encoding="utf-8-sig")
            t = out[out["split"] == "test"]
            metrics = {
                "target": target, "model": model_name or "tfidf+logreg", "classes": classes,
                "date": dt.datetime.now().strftime("%Y-%m-%d %H:%M"), "filters": FILTERS,
                "train": len(train), "val": len(val),
                "scores": {s: scores(out.loc[out["split"] == s, "label"], out.loc[out["split"] == s, "pred"])
                           for s in EVAL_SPLITS},
                "per_class": per_class(t).to_dict("records"),
                "per_lang": [{"lang": l, **scores(g["label"], g["pred"])} for l, g in t.groupby("lang")],
            }
            with open(os.path.join(mdir, "metrics.json"), "w") as f:
                json.dump(metrics, f, indent=2, ensure_ascii=False)
            out[out["label"] != out["pred"]].to_csv(os.path.join(mdir, "errors.csv"), index=False,
                                                    encoding="utf-8-sig")
        for s in EVAL_SPLITS:
            g = out[out["split"] == s]
            if len(g):
                summary.append({"model": name, "split": s, **scores(g["label"], g["pred"])})

    # The AI pre-label against the final label.
    g = ev[(ev["split"] == "test") & ev["claude"].notna()]
    if len(g):
        summary.append({"model": "ai_suggestion", "split": "test", **scores(g["label"], g["claude"])})

    summary = pd.DataFrame(summary)
    summary.to_csv(os.path.join(tdir, "summary.csv"), index=False)
    test_scores = summary[(summary["split"] == "test") & (summary["model"].isin(runs))]
    best = test_scores.sort_values("macro_f1").iloc[-1]["model"]
    best_pred = pd.read_csv(os.path.join(tdir, best, "predictions.csv"))
    t = best_pred[best_pred["split"] == "test"]
    confusion = pd.crosstab(t["label"], t["pred"], rownames=["real"], colnames=["predicted"])
    md = f"""# {RUN_NAME} · {target}

Filters: `{FILTERS or "none"}` · classes: {len(classes)}
Splits: {", ".join(f"{s}={n}" for s, n in data["split"].value_counts().items())}

`test` = 20% of the notes, never seen in training.
`ai_suggestion` = the AI pre-label (claude_{target}) against the final label.

## Summary

{md_table(summary)}

## Per class, best model on test: {best}

{md_table(per_class(t))}

## Confusion matrix ({best}, test, rows = real)

{md_table(confusion.reset_index())}
"""
    with open(os.path.join(tdir, "report.md"), "w") as f:
        f.write(md)
    print(f"Saved to {tdir} · best on test: {best}\n")
    return tdir

## 6. Train

On an L4: ~10 min per target with mDeBERTa base and ~30 min with XLM-R large
(A100: ~2x faster). At the end, models and results are copied to Drive.

Training writes to the Colab disk. If Colab disconnects before the copy, the
run is lost: run all cells again.

In [ ]:
run_cat = train_target("category")
run_theme = train_target("theme")

# Copy models and results to Drive (no checkpoints).
!mkdir -p "{DRIVE_OUT}" && rsync -a --exclude checkpoints "{OUT}/" "{DRIVE_OUT}/"
print("Copied to", os.path.join(DRIVE_OUT, RUN_NAME))

## 7. Results

- `test`: 20% of the notes, never seen in training.
- `ai_suggestion`: the AI pre-label against the same final labels.
- **macro-F1**: the average F1 over classes, so rare classes count as much as
  common ones.

In [ ]:
import matplotlib.pyplot as plt


def model_dirs(tdir):
    names = ["baseline"] + [short(m) for m in MODELS]
    return {n: os.path.join(tdir, n) for n in names if os.path.isfile(os.path.join(tdir, n, "metrics.json"))}


def show(tdir):
    print(f"== {tdir}")
    summary = pd.read_csv(os.path.join(tdir, "summary.csv"))
    display(summary.pivot(index="model", columns="split", values="macro_f1").round(3))

    dirs = model_dirs(tdir)
    fig, ax = plt.subplots(1, 2, figsize=(18, 5))
    summary[summary["split"] == "test"].set_index("model")[["macro_f1", "accuracy"]] \
        .plot.bar(ax=ax[0], ylim=(0, 1), rot=0, title="macro-F1 and accuracy by model (test)")
    f1 = pd.DataFrame({n: pd.DataFrame(json.load(open(f"{d}/metrics.json"))["per_class"]).set_index("class")["f1"]
                       for n, d in dirs.items()})
    f1.plot.barh(ax=ax[1], xlim=(0, 1), title="F1 by class (test)")
    plt.tight_layout(); plt.show()

    best = summary[(summary["split"] == "test") & summary["model"].isin(dirs)].sort_values("macro_f1").iloc[-1]["model"]
    pred = pd.read_csv(f"{dirs[best]}/predictions.csv")
    t = pred[pred["split"] == "test"]
    cm = pd.crosstab(t["label"], t["pred"])
    fig, a = plt.subplots(figsize=(9, 7))
    a.imshow(cm.values, cmap="Blues")
    a.set_xticks(range(len(cm.columns)), cm.columns, rotation=60, ha="right")
    a.set_yticks(range(len(cm.index)), cm.index)
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            a.text(j, i, cm.values[i, j], ha="center", va="center", fontsize=8)
    a.set_title(f"Confusion matrix · {best} (test, rows = true label)")
    plt.tight_layout(); plt.show()

    best_metrics = json.load(open(f"{dirs[best]}/metrics.json"))
    print("By language (test):"); display(pd.DataFrame(best_metrics["per_lang"]))
    support = pd.DataFrame(best_metrics["per_class"]).set_index("class")["support"]
    print("F1 by class and model (test):"); display(f1.assign(support=support).round(2))
    return best


best_cat = show(run_cat)
best_theme = show(run_theme)

### Confidence: when to send a note to a person?

If the model's confidence is below the threshold, the note goes to human review
(this replaces `unsure`). `coverage` = share of notes the model decides on its own.

In [ ]:
def by_confidence(tdir, best):
    t = pd.read_csv(os.path.join(tdir, best, "predictions.csv"))
    t = t[t["split"] == "test"]
    rows = []
    for th in (0.0, 0.5, 0.6, 0.7, 0.8, 0.9):
        g = t[t["conf"] >= th]
        rows.append({"threshold": th, "coverage": round(len(g) / len(t), 2),
                     "accuracy": round((g["label"] == g["pred"]).mean(), 3) if len(g) else None})
    return pd.DataFrame(rows)


def by_length(tdir, best):
    t = pd.read_csv(os.path.join(tdir, best, "predictions.csv"))
    t = t[t["split"] == "test"].copy()
    t["words"] = t["user_text"].fillna("").str.split().str.len()
    t["length"] = pd.cut(t["words"], [-1, 0, 3, 10, 30, 10_000], labels=["0", "1-3", "4-10", "11-30", ">30"])
    return t.assign(ok=t["label"] == t["pred"]).groupby("length", observed=False)["ok"] \
            .agg(["count", "mean"]).rename(columns={"count": "n", "mean": "accuracy"}).round(2)


print(f"Category · {best_cat}"); display(by_confidence(run_cat, best_cat)); display(by_length(run_cat, best_cat))
print(f"Theme · {best_theme}"); display(by_confidence(run_theme, best_theme)); display(by_length(run_theme, best_theme))

## 8. Errors

Wrong predictions of the best category model, the most confident first.

In [ ]:
err = pd.read_csv(os.path.join(run_cat, best_cat, "errors.csv"))
pd.set_option("display.max_colwidth", 200)
err[["note_id", "split", "lang", "label", "pred", "conf", "claude", "text"]] \
    .sort_values("conf", ascending=False).head(50)

## 9. Download

The models are already in Drive (`DRIVE_OUT/RUN_NAME/<target>/<model>/model/`).
This downloads only reports and predictions. `WITH_MODELS = True` adds the
models (~1 GB for base, ~2 GB for large, each).

In [ ]:
from google.colab import files
WITH_MODELS = False
exclude = '-x "*/checkpoints/*"' + ('' if WITH_MODELS else ' -x "*/model/*"')
!cd "{OUT}" && zip -qr /content/{RUN_NAME}.zip "{RUN_NAME}" {exclude}
files.download(f"/content/{RUN_NAME}.zip")